# Federated Learning for Brain Tumor Classification with PIDL

This notebook runs the **full pipeline** with **real cryptographic secure aggregation** (Flower SecAgg+).

- **ResNet-18** backbone + **PIDL loss** (Perona-Malik regularization)
- **True encryption**: Flower SecAgg+ (secret-sharing, no simulation)
- Stratified data split, centralized evaluation on a held-out test set
- Logs saved to CSV for plotting

**Flow**: Mount Drive → Clone repo → Install → Run `flwr run .` (SecAgg+) → Plot results.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository (use your GitHub URL) and go into project root
# If you already have the repo under /content, skip clone and set PROJECT_DIR accordingly.
import os
REPO_URL = "https://github.com/PulockDas/brain-tumor-classification-PIDL-FL.git"  # ← set your repo URL
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} /content/brain-tumor-classification-PIDL-FL
%cd {PROJECT_DIR}

In [ ]:
# Install project and dependencies (includes flwr[simulation]>=1.25.0 for SecAgg+)
# Run from project root so "pip install -e ." finds pyproject.toml
!cd {PROJECT_DIR} && pip install -e .

## Configuration

In [ ]:
# Configuration — used for flwr run and for plotting
DATA_ROOT = "/content/drive/MyDrive/PhysNet/datasets/brain_tumor_mri"
# Use local dir to avoid filling Drive; copy results to repo later if needed (see README)
LOG_DIR   = "/content/results"
NUM_ROUNDS = 10
LOCAL_EPOCHS = 5
MIN_CLIENTS = 3

## Run Federated Learning

In [ ]:
# Run federated learning with real cryptographic secure aggregation (Flower SecAgg+)
# Logs to LOG_DIR: fl_rounds.csv, fl_clients.csv, fl_eval.json, fl_summary.json (no Drive).
import os

os.makedirs(LOG_DIR, exist_ok=True)
# Flower expects one --run-config string: 'k1=v1 k2=v2 ...' (not multiple args)
_run_config = (
    f'data-root="{DATA_ROOT}" '
    f'log-dir="{LOG_DIR}" '
    f'num-server-rounds={NUM_ROUNDS} '
    f'local-epochs={LOCAL_EPOCHS} '
    f'min-fit-clients={MIN_CLIENTS} '
    'is-demo=false'
)
# Run in the notebook shell so all output and errors stream here. PYTHONPATH so workers see data, models, etc.
_cmd = f'cd "{PROJECT_DIR}" && PYTHONPATH="{PROJECT_DIR}" flwr run . -c \'{_run_config}\''
_exit_code = get_ipython().system(_cmd)
if _exit_code != 0:
    raise SystemExit(f"flwr run exited with code {_exit_code}. Check the log above for errors.")
print(f"\n→ Results (fl_rounds.csv, fl_clients.csv, fl_eval.json, fl_summary.json) are in: {LOG_DIR}")

## Plot Results

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

log_dir = LOG_DIR
rounds_path = os.path.join(log_dir, "fl_rounds.csv")
clients_path = os.path.join(log_dir, "fl_clients.csv")
eval_path = os.path.join(log_dir, "fl_eval.json")
summary_path = os.path.join(log_dir, "fl_summary.json")

if not os.path.isfile(rounds_path) or not os.path.isfile(clients_path):
    print("No results yet. Run the previous cell (flwr run) first. If you already ran it, check LOG_DIR and that the SecAgg+ app wrote CSVs there.")
else:
    rounds_df = pd.read_csv(rounds_path)
    clients_df = pd.read_csv(clients_path)

    # Global test accuracy over rounds
    plt.figure(figsize=(10, 6))
    plt.plot(rounds_df["round"], rounds_df["global_test_acc"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Global Test Accuracy over FL Rounds (SecAgg+)")
    plt.grid(True)
    plt.show()

    # F1 macro over rounds (if present)
    if "f1_macro" in rounds_df.columns:
        plt.figure(figsize=(10, 6))
        plt.plot(rounds_df["round"], rounds_df["f1_macro"], marker="o", linewidth=2, color="green")
        plt.xlabel("Round")
        plt.ylabel("F1 (macro)")
        plt.title("Global Test F1 (macro) over FL Rounds")
        plt.grid(True)
        plt.show()

    # Inference and training time per round (if present)
    if "inference_time_sec" in rounds_df.columns and "training_time_sec" in rounds_df.columns:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(rounds_df["round"], rounds_df["inference_time_sec"], marker="o", label="Inference time (s)", linewidth=2)
        ax.plot(rounds_df["round"], rounds_df["training_time_sec"], marker="s", label="Training time (s)", linewidth=2)
        ax.set_xlabel("Round")
        ax.set_ylabel("Time (s)")
        ax.set_title("Inference & Training Time per Round")
        ax.legend()
        ax.grid(True)
        plt.tight_layout()
        plt.show()

    # Client training accuracies
    plt.figure(figsize=(12, 6))
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_acc"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Accuracy (%)")
    plt.title("Client Training Accuracies over FL Rounds")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Losses
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(rounds_df["round"], rounds_df["global_test_loss"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Loss")
    plt.title("Global Test Loss")
    plt.grid(True)
    plt.subplot(1, 2, 2)
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_loss"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Loss")
    plt.title("Client Training Losses")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Final confusion matrix (from fl_eval.json, last round)
    if os.path.isfile(eval_path):
        with open(eval_path) as f:
            ev = json.load(f)
        rounds_ev = ev.get("rounds", [])
        class_names = ev.get("class_names") or [f"C{i}" for i in range(4)]
        if rounds_ev and "confusion_matrix" in rounds_ev[-1]:
            cm = np.array(rounds_ev[-1]["confusion_matrix"])
            plt.figure(figsize=(8, 6))
            plt.imshow(cm, interpolation="nearest", cmap="Blues")
            plt.colorbar()
            plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
            plt.yticks(np.arange(len(class_names)), class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title("Confusion Matrix (final round)")
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, int(cm[i, j]), ha="center", va="center", color="black" if cm[i, j] < cm.max() / 2 else "white")
            plt.tight_layout()
            plt.show()

    if os.path.isfile(summary_path):
        with open(summary_path) as f:
            s = json.load(f)
        print("Summary:", json.dumps(s, indent=2))

## Push result files to GitHub

Run this cell after FL finishes. It copies the important result files into the repo, commits them, and pushes to GitHub.

In [ ]:
import os
import shutil

RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
FILES = ["fl_rounds.csv", "fl_clients.csv", "fl_eval.json", "config.json", "fl_summary.json"]

os.makedirs(RESULTS_DIR, exist_ok=True)
copied = []
for f in FILES:
    src = os.path.join(LOG_DIR, f)
    dst = os.path.join(RESULTS_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        copied.append(f)
    else:
        print(f"Skipping {f} (not found in {LOG_DIR})")

if not copied:
    print("No result files to push. Run the FL cell first.")
else:
    print(f"Copied to repo: {copied}")
    add_list = " ".join(os.path.join("results", f) for f in copied)
    !cd {PROJECT_DIR} && git add {add_list} && git status
    !cd {PROJECT_DIR} && git commit -m "Add FL result artifacts" || true
    !cd {PROJECT_DIR} && git push origin HEAD
    print("Pushed to GitHub.")